# TP1: Introduction to DataFrame (according to spark)


## Quick guide to using DataFrame

https://spark.apache.org/docs/latest/sql-programming-guide.html

python api that lists all the functions applicable to a dataframe.

https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.DataFrame.html

The aim of this tutorial is to start a spark session and manipulate dataframes by applying basic processing.

1. Start a pyspark session

2. Create a dataframe from an RDD or a list (don't forget to define a structure).

3. Create a pandas dataframe from a dataframe (spark)

4. Perform operations (compare with list processing)

5. Grouping and joins

6. Reading a csv
 
 https://notebooks.gesis.org/binder/jupyter/user/apache-spark-awl6064c/notebooks/python/docs/source/getting_started/quickstart_df.ipynb


### 1. Start a spark session

(we want to start a local session and give it a name).

We'll use the SparkSession class from the [*pyspark.sql*](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/index.html) sub-module .

Call *sc* the *spark.sparkContext*.

- *builder*: A class attribute having a Builder to construct SparkSession instances

- Specify what is defined by [*SparkSession*](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/spark_session.html) and *sparkContext*.

- the master is 'local' (*master("local[\*])* (it is possible to specify the number of cores *local[\*]* or for example *local[4]*) 

- the application must be given a name *appName('name appli')*.

- *getOrCreate()* : Gets an existing SparkSession or, if there is no existing one, creates a new one based on the options set in this builder.

In [1]:
from datetime import datetime, date
import pandas as pd

# import pyspark and sparksession
import pyspark
from pyspark.sql import SparkSession

from pyspark.sql.types import *
from pyspark.sql import Row

# link pyspark and spark
import findspark
findspark.init()

In [2]:
# define spark session
spark = SparkSession.builder.master("local[*]").appName('TP1_dataframe').getOrCreate()
# create sparkContext variable
sc = spark.sparkContext

In [3]:
# display sparkContext
sc

<SparkContext master=local[*] appName=TP1_dataframe>

In [33]:
# pypsark version == spark version
pyspark.__version__

'3.5.0'

A port (4040 by default) that you should definitely look at (link Spark UI)

It allows you to follow the progress of a spark process.

http://localhost:4040



[*.parallelize()*](https://spark.apache.org/docs/latest/api/python/reference/api/pyspark.SparkContext.parallelize.html) : Distribute a local Python collection to form an RDD

### 1. Opération sur un Dataframe

- selectionnner des colonnes

- ajouter des nouvelles colonnes en faisant un traitement


- créer un dataframe (panda) de 4 éléments composé de 4 colonnes (1ère colonne de type int, 2ème de type string, 3ème colonne une date, 4ème colonne une valeur de type float)

- nommer les colonnes

In [18]:
pandas_df = pd.DataFrame({
    'indice': [1, 2, 3, 4],
    'prenom': ['john', 'jean', 'pierre', 'jacques'],
    'nom': ['doe', 'dupond', 'smith', 'durand'],
    'date': [date(2000, 1, 1), date(2000, 2, 1), date(2000, 3, 1), date(2000, 4, 1)],
    'valeur': [1., 2., 3., 4.]
})
df = spark.createDataFrame(pandas_df)

df.describe().show()

df.show()

+-------+------------------+-------+-----+------------------+
|summary|            indice| prenom|  nom|            valeur|
+-------+------------------+-------+-----+------------------+
|  count|                 4|      4|    4|                 4|
|   mean|               2.5|   NULL| NULL|               2.5|
| stddev|1.2909944487358056|   NULL| NULL|1.2909944487358056|
|    min|                 1|jacques|  doe|               1.0|
|    max|                 4| pierre|smith|               4.0|
+-------+------------------+-------+-----+------------------+

+------+-------+------+----------+------+
|indice| prenom|   nom|      date|valeur|
+------+-------+------+----------+------+
|     1|   john|   doe|2000-01-01|   1.0|
|     2|   jean|dupond|2000-02-01|   2.0|
|     3| pierre| smith|2000-03-01|   3.0|
|     4|jacques|durand|2000-04-01|   4.0|
+------+-------+------+----------+------+



- ajouter une 5ème colonne qui indique le type des données de la 2 ème colonne

- ajouter une 6ème colonne qui est le résultat d'un opération arithmétique sur la 4ème colonne
- afficher le résultat

In [19]:
df.withColumn("operation",df.valeur +3).show()

from pyspark.sql.functions import concat_ws

df.withColumn("nom+prenom", concat_ws(" ","prenom","nom")).show()

+------+-------+------+----------+------+---------+
|indice| prenom|   nom|      date|valeur|operation|
+------+-------+------+----------+------+---------+
|     1|   john|   doe|2000-01-01|   1.0|      4.0|
|     2|   jean|dupond|2000-02-01|   2.0|      5.0|
|     3| pierre| smith|2000-03-01|   3.0|      6.0|
|     4|jacques|durand|2000-04-01|   4.0|      7.0|
+------+-------+------+----------+------+---------+

+------+-------+------+----------+------+--------------+
|indice| prenom|   nom|      date|valeur|    nom+prenom|
+------+-------+------+----------+------+--------------+
|     1|   john|   doe|2000-01-01|   1.0|      john doe|
|     2|   jean|dupond|2000-02-01|   2.0|   jean dupond|
|     3| pierre| smith|2000-03-01|   3.0|  pierre smith|
|     4|jacques|durand|2000-04-01|   4.0|jacques durand|
+------+-------+------+----------+------+--------------+



- application d'une opération sur la colonne en utilisant une fonction

(des fonctions pandas peuvent être appliquées)
https://spark.apache.org/docs/latest/api/python/user_guide/sql/arrow_pandas.html

In [20]:
from pyspark.sql.functions import col, pandas_udf
from pyspark.sql.types import LongType
from pyspark.sql.functions import udf


# Declare the function and create the UDF
def multiply_func(a: pd.Series, b: pd.Series) -> pd.Series:
    return a * b

@udf("float") 
def tripled(num):
  return 3*float(num)

multiply = pandas_udf(multiply_func, returnType=FloatType())

# Execute function as a Spark vectorized UDF
df.select("*",multiply(col("valeur"), col("valeur"))).show()

df.withColumn('tripled_col', tripled(df.valeur)).show()


+------+-------+------+----------+------+-----------------------------+
|indice| prenom|   nom|      date|valeur|multiply_func(valeur, valeur)|
+------+-------+------+----------+------+-----------------------------+
|     1|   john|   doe|2000-01-01|   1.0|                          1.0|
|     2|   jean|dupond|2000-02-01|   2.0|                          4.0|
|     3| pierre| smith|2000-03-01|   3.0|                          9.0|
|     4|jacques|durand|2000-04-01|   4.0|                         16.0|
+------+-------+------+----------+------+-----------------------------+

+------+-------+------+----------+------+-----------+
|indice| prenom|   nom|      date|valeur|tripled_col|
+------+-------+------+----------+------+-----------+
|     1|   john|   doe|2000-01-01|   1.0|        3.0|
|     2|   jean|dupond|2000-02-01|   2.0|        6.0|
|     3| pierre| smith|2000-03-01|   3.0|        9.0|
|     4|jacques|durand|2000-04-01|   4.0|       12.0|
+------+-------+------+----------+------+----

### 2. Le retour du regroupement mais appliqué au dataframe

In [21]:
# soit le dataframe suivant

df = spark.createDataFrame([
    ['red', 'banana', 1, 10], ['blue', 'banana', 2, 20], ['red', 'carrot', 3, 30],
    ['blue', 'grape', 4, 40], ['red', 'carrot', 5, 50], ['black', 'carrot', 6, 60],
    ['red', 'banana', 7, 70], ['red', 'grape', 8, 80]], schema=['color', 'fruit', 'v1', 'v2'])


- on regroupe par couleur et on calcul la valeur moyenne et le nombre total d'élément

In [22]:
df1 = df.groupBy("color","fruit").mean("v1","v2").show()


+-----+------+-------+-------+
|color| fruit|avg(v1)|avg(v2)|
+-----+------+-------+-------+
|  red|banana|    4.0|   40.0|
| blue|banana|    2.0|   20.0|
| blue| grape|    4.0|   40.0|
|  red|carrot|    4.0|   40.0|
|black|carrot|    6.0|   60.0|
|  red| grape|    8.0|   80.0|
+-----+------+-------+-------+



fusion et ajout

In [23]:
# soit les deux dataframes suivants :
df1 = spark.createDataFrame(
    [(20000101, 1, 1.0), (20000101, 2, 2.0), (20000102, 1, 3.0), (20000102, 2, 4.0)],
    ('time', 'id', 'v1'))

df2 = spark.createDataFrame(
    [(20000101, 1, 'x'), (20000101, 2, 'y')],
    ('time', 'id', 'v2'))

# utiliser inner, left, right,...

df1.join(df2,df1.time ==  df2.time,"inner") \
     .show(truncate=False)


+--------+---+---+--------+---+---+
|time    |id |v1 |time    |id |v2 |
+--------+---+---+--------+---+---+
|20000101|1  |1.0|20000101|1  |x  |
|20000101|1  |1.0|20000101|2  |y  |
|20000101|2  |2.0|20000101|1  |x  |
|20000101|2  |2.0|20000101|2  |y  |
+--------+---+---+--------+---+---+



Effectuer une regroupement qui permet de créer un dataframe à 4 colonnes (time id v1 v2)

In [24]:
# les deux colonnes time sont identiques

df1.join(df2,["time"],"inner") \
     .show(truncate=False)

+--------+---+---+---+---+
|time    |id |v1 |id |v2 |
+--------+---+---+---+---+
|20000101|1  |1.0|1  |x  |
|20000101|1  |1.0|2  |y  |
|20000101|2  |2.0|1  |x  |
|20000101|2  |2.0|2  |y  |
+--------+---+---+---+---+



In [26]:
sc.stop()